# Plain CTS + Scaife Search and URN Navigation

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction and scope</a>
* <a href="#search-types">2 - What can be searched?</a>
* <a href="#setup">3 - Plain HTTP setup and helpers</a>
* <a href="#cts-inventory">4 - Load the plain CTS inventory</a>
* <a href="#author-search">5 - Search for authors and textgroups</a>
* <a href="#work-search">6 - Search for works and books</a>
* <a href="#form-search">7 - Search for a surface word form</a>
* <a href="#lemma-search">8 - Search by lemma</a>
* <a href="#phrase-search">9 - Search for an exact phrase</a>
* <a href="#scoped-search">10 - Scope a search by author or work</a>
* <a href="#urn-mismatch">11 - Demonstrate the CTS and Scaife URN mismatch</a>
* <a href="#navigate-hit">12 - Navigate around a search hit with plain CTS</a>
* <a href="#decision-guide">13 - Search decision guide and MCP equivalents</a>
* <a href="#next-steps">14 - Continue learning</a>
* <a href="#sources">15 - Sources</a>
* <a href="#required-libraries">16 - Required libraries</a>
* <a href="#notebook-version">17 - Notebook version</a>

## 1 - Introduction and scope <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook explains several different meanings of **searching Perseus**: finding authors, finding works, finding an exact written form, finding all forms associated with a lemma, searching a phrase, narrowing a query to an author or work, and navigating around a result.

> **Important scope:** this is a **plain HTTP notebook**, not an MCP notebook. It does not import `perseus_mcp.server`, FastMCP, or an MCP client, and it does not call any MCP tool. Author/work discovery and citation navigation use the upstream **Perseus CTS endpoint** directly. Full-text word, lemma, phrase, and scoped searches use the upstream **Scaife search API** directly.

The MCP tool names appear only near the end as a comparison. The code in this notebook shows the lower-level services beneath those tools.

By the end of the notebook you will be able to:

- distinguish metadata search from full-text search;
- search the CTS inventory for partial and exact author names;
- search the CTS inventory for work titles and available editions;
- distinguish a surface-form search from a lemma search;
- search a quoted phrase and apply author/work scopes;
- see why CTS and Scaife edition URNs may differ for the same work;
- translate a Scaife hit into the separately discovered CTS edition before navigating.

## 2 - What can be searched? <a class="anchor" id="search-types"></a>
##### [Back to ToC](#TOC)

The word *search* hides several different operations. Choosing the correct data source is the first step.

| Research question | What is searched? | Plain service used here |
|---|---|---|
| "Is Homer in the inventory?" | Author/textgroup names and URNs | CTS `GetCapabilities` |
| "Which works are attributed to Plato?" | Works nested below a textgroup | CTS `GetCapabilities` |
| "Which work is titled Republic?" | Work titles and work URNs | CTS `GetCapabilities` |
| "Where does the written form μῆνιν occur?" | Surface tokens in indexed text | Scaife library search with `kind=form` |
| "Where do forms of the lemma λόγος occur?" | Lemma annotations in indexed text | Scaife library search with `kind=lemma` |
| "Where does the phrase μῆνιν ἄειδε occur?" | Adjacent text matching a quoted query | Scaife library search |
| "Only in Homer or the Iliad?" | Search results constrained by CTS textgroup/work URNs | Scaife `text_group` or `work` scope |
| "What comes before and after this hit?" | Ordered valid citation URNs | CTS `GetValidReff` |

Two distinctions are especially important:

1. **CTS is not a general full-text search engine.** It supplies inventory, resource, passage, and citation services. In this notebook, searching authors and books means downloading `GetCapabilities` and filtering its metadata locally.
2. **A classical "book" can mean two things.** A title such as *Iliad* or *Republic* is a CTS **work**. "Book 1 of the Iliad" is an internal citation level and is reached through passage references, not work-title search.

## 3 - Plain HTTP setup and helpers <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

Only `httpx` is installed. The remaining modules are part of Python's standard library. The helpers below make direct requests, remove XML namespaces, parse CTS metadata, and turn Scaife results into compact rows.

Again, there is no MCP connection in this setup.

In [4]:
%pip install --quiet httpx

Note: you may need to restart the kernel to use updated packages.


In [5]:
import html
import json
import re
import unicodedata
import xml.etree.ElementTree as ET

import httpx

CTS_BASE = "https://www.perseus.tufts.edu/hopper/CTS"
SCAIFE_SEARCH = "https://scaife.perseus.org/search/json/"
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
TAG_RE = re.compile(r"<[^>]+>")


def cts_request(request, urn=None, timeout=60.0, **extra_params):
    """Call the plain Perseus CTS endpoint and return response text."""
    params = {"request": request, **extra_params}
    if urn is not None:
        params["urn"] = urn

    response = httpx.get(
        CTS_BASE,
        params=params,
        timeout=timeout,
        follow_redirects=True,
    )
    response.raise_for_status()
    return response.text


def scaife_search(
    query,
    *,
    kind="form",
    page_num=1,
    text_group=None,
    work=None,
    result_format="instances",
):
    """Call the plain Scaife library-search endpoint and return JSON."""
    params = {
        "q": unicodedata.normalize("NFC", query),
        "kind": kind,
        "type": "library",
        "page_num": page_num,
        "format": result_format,
    }
    if text_group:
        params["text_group"] = text_group
    if work:
        params["work"] = work

    response = httpx.get(
        SCAIFE_SEARCH,
        params=params,
        timeout=30.0,
        follow_redirects=True,
    )
    response.raise_for_status()
    return response.json()


def local_name(tag):
    return tag.rsplit("}", 1)[-1]


def element_text(element):
    return " ".join("".join(element.itertext()).split())


def direct_children(element, name):
    wanted = name.casefold()
    return [child for child in list(element) if local_name(child.tag).casefold() == wanted]


def direct_texts(element, name):
    return [
        element_text(child)
        for child in direct_children(element, name)
        if element_text(child)
    ]


def clean_snippet(content):
    joined = " | ".join(content or [])
    return " ".join(html.unescape(TAG_RE.sub("", joined)).split())


def search_rows(data, limit=5):
    rows = []
    for result in data.get("results", [])[:limit]:
        passage = result.get("passage", {})
        text = passage.get("text", {})
        ancestors = text.get("ancestors", [])
        rows.append(
            {
                "passage_urn": passage.get("urn"),
                "text_urn": text.get("urn"),
                "author": ancestors[0].get("label") if len(ancestors) > 0 else None,
                "work": ancestors[1].get("label") if len(ancestors) > 1 else None,
                "reference": passage.get("refs", {}).get("start", {}).get("human_reference"),
                "snippet": clean_snippet(result.get("content")),
            }
        )
    return rows


def print_search_summary(label, data, limit=3):
    print(f"\n{label}")
    print(f"query={data.get('q')!r}, kind={data.get('kind')!r}, total_count={data.get('total_count')}")
    print(json.dumps(search_rows(data, limit), ensure_ascii=False, indent=2))

## 4 - Load the plain CTS inventory <a class="anchor" id="cts-inventory"></a>
##### [Back to ToC](#TOC)

`GetCapabilities` is the plain CTS inventory document. It describes textgroups/authors, works, editions, and translations. The document is currently about two megabytes, so the notebook downloads and parses it once.

The parser creates an ordinary Python catalog. The later author and work "searches" are local filters over this catalog—not calls to a special CTS search endpoint.

In [6]:
capabilities_xml = cts_request("GetCapabilities")
capabilities_root = ET.fromstring(capabilities_xml)


def parse_cts_catalog(root):
    catalog = []

    for text_group in root.iter():
        if local_name(text_group.tag).casefold() != "textgroup":
            continue

        author = {
            "urn": text_group.attrib.get("urn"),
            "names": direct_texts(text_group, "groupname"),
            "works": [],
        }

        for work_element in direct_children(text_group, "work"):
            work_language = work_element.attrib.get(XML_LANG) or work_element.attrib.get("xml:lang")
            work = {
                "urn": work_element.attrib.get("urn"),
                "language": work_language,
                "titles": direct_texts(work_element, "title"),
                "resources": [],
            }

            for resource_element in list(work_element):
                resource_type = local_name(resource_element.tag).casefold()
                if resource_type not in {"edition", "translation"}:
                    continue
                work["resources"].append(
                    {
                        "type": resource_type,
                        "urn": resource_element.attrib.get("urn"),
                        "language": (
                            resource_element.attrib.get(XML_LANG)
                            or resource_element.attrib.get("xml:lang")
                            or work_language
                        ),
                        "labels": (
                            direct_texts(resource_element, "label")
                            or direct_texts(resource_element, "title")
                        ),
                    }
                )

            author["works"].append(work)

        catalog.append(author)

    return catalog


cts_catalog = parse_cts_catalog(capabilities_root)
print(f"Downloaded bytes: {len(capabilities_xml.encode('utf-8')):,}")
print(f"CTS textgroups/authors: {len(cts_catalog):,}")
print(f"CTS works: {sum(len(author['works']) for author in cts_catalog):,}")

Downloaded bytes: 2,052,315
CTS textgroups/authors: 140
CTS works: 1,179


## 5 - Search for authors and textgroups <a class="anchor" id="author-search"></a>
##### [Back to ToC](#TOC)

CTS calls the author-level container a **textgroup**. A name search should usually allow partial matches for discovery but rank exact matches first.

The query `Hom` demonstrates why partial search and exact identity are different: it can match both `Homer` and `Homeric Hymns`. Downstream code should keep the URN, not only the displayed name.

In [7]:
def search_authors(query):
    needle = " ".join(query.split()).casefold()
    matches = []

    for author in cts_catalog:
        normalized_names = [name.casefold() for name in author["names"]]
        if needle not in " ".join(normalized_names) and needle not in (author["urn"] or "").casefold():
            continue
        matches.append(
            {
                "urn": author["urn"],
                "names": author["names"],
                "works_count": len(author["works"]),
                "sample_works": [
                    work["titles"][0] if work["titles"] else work["urn"]
                    for work in author["works"][:5]
                ],
                "exact_name_match": needle in normalized_names,
            }
        )

    return sorted(matches, key=lambda row: (not row["exact_name_match"], row["names"]))


partial_homer_matches = search_authors("Hom")
exact_homer_matches = search_authors("Homer")

print("Partial author query: 'Hom'")
print(json.dumps(partial_homer_matches, ensure_ascii=False, indent=2))

print("\nExact-name query ranked first: 'Homer'")
print(json.dumps(exact_homer_matches[:3], ensure_ascii=False, indent=2))

HOMER_TEXTGROUP = next(
    row["urn"]
    for row in exact_homer_matches
    if row["exact_name_match"] and "Homer" in row["names"]
)
print(f"\nSelected Homer textgroup: {HOMER_TEXTGROUP}")

Partial author query: 'Hom'
[
  {
    "urn": "urn:cts:greekLit:tlg0012",
    "names": [
      "Homer"
    ],
    "works_count": 2,
    "sample_works": [
      "Iliad",
      "Odyssey"
    ],
    "exact_name_match": false
  },
  {
    "urn": "urn:cts:greekLit:tlg0013",
    "names": [
      "Homeric Hymns"
    ],
    "works_count": 33,
    "sample_works": [
      "Hymn 27 to Artemis",
      "Hymn 26 to Dionysus",
      "Hymn 1 to Dionysus",
      "Hymn 29 to Hestia",
      "Hymn 2 to Demeter"
    ],
    "exact_name_match": false
  }
]

Exact-name query ranked first: 'Homer'
[
  {
    "urn": "urn:cts:greekLit:tlg0012",
    "names": [
      "Homer"
    ],
    "works_count": 2,
    "sample_works": [
      "Iliad",
      "Odyssey"
    ],
    "exact_name_match": true
  },
  {
    "urn": "urn:cts:greekLit:tlg0013",
    "names": [
      "Homeric Hymns"
    ],
    "works_count": 33,
    "sample_works": [
      "Hymn 27 to Artemis",
      "Hymn 26 to Dionysus",
      "Hymn 1 to Dionysus",
      "

## 6 - Search for works and books <a class="anchor" id="work-search"></a>
##### [Back to ToC](#TOC)

A CTS **work** is the level used for titles such as *Iliad*, *Odyssey*, or *Republic*. The work may have several resources below it: Greek editions, Latin editions, or translations.

The first example searches for *Republic* across the inventory. The second selects the exact *Iliad* work and then selects a Greek CTS edition advertised beneath it. That edition will later be compared with the edition returned by Scaife search.

In [8]:
def search_works(query):
    needle = " ".join(query.split()).casefold()
    matches = []

    for author in cts_catalog:
        for work in author["works"]:
            normalized_titles = [title.casefold() for title in work["titles"]]
            searchable = " ".join([work["urn"] or "", *work["titles"]]).casefold()
            if needle not in searchable:
                continue
            matches.append(
                {
                    "author_urn": author["urn"],
                    "author_names": author["names"],
                    "work_urn": work["urn"],
                    "titles": work["titles"],
                    "language": work["language"],
                    "resources": work["resources"],
                    "exact_title_match": needle in normalized_titles,
                }
            )

    return sorted(
        matches,
        key=lambda row: (not row["exact_title_match"], row["author_names"], row["titles"]),
    )


republic_matches = search_works("Republic")
print("Works matching 'Republic':")
print(json.dumps(republic_matches[:5], ensure_ascii=False, indent=2))

iliad_match = next(
    row
    for row in search_works("Iliad")
    if row["work_urn"].startswith(HOMER_TEXTGROUP) and row["exact_title_match"]
)
ILIAD_WORK = iliad_match["work_urn"]
cts_greek_editions = [
    resource
    for resource in iliad_match["resources"]
    if resource["type"] == "edition" and resource["language"] == "grc"
]
if not cts_greek_editions:
    raise RuntimeError("The live CTS inventory advertised no Greek Iliad edition.")

CTS_ILIAD_EDITION = cts_greek_editions[0]["urn"]
print("\nSelected CTS resource:")
print(
    json.dumps(
        {
            "author": iliad_match["author_names"],
            "work": iliad_match["titles"],
            "work_urn": ILIAD_WORK,
            "greek_editions": cts_greek_editions,
            "selected_cts_edition": CTS_ILIAD_EDITION,
        },
        ensure_ascii=False,
        indent=2,
    )
)

Works matching 'Republic':
[
  {
    "author_urn": "urn:cts:greekLit:tlg0059",
    "author_names": [
      "Plato"
    ],
    "work_urn": "urn:cts:greekLit:tlg0059.tlg030",
    "titles": [
      "Republic"
    ],
    "language": "grc",
    "resources": [
      {
        "type": "edition",
        "urn": "urn:cts:greekLit:tlg0059.tlg030.perseus-grc1",
        "language": "grc",
        "labels": [
          "Republic"
        ]
      },
      {
        "type": "translation",
        "urn": "urn:cts:greekLit:tlg0059.tlg030.perseus-eng1",
        "language": "eng",
        "labels": [
          "Republic"
        ]
      }
    ],
    "exact_title_match": true
  },
  {
    "author_urn": "urn:cts:latinLit:phi0474",
    "author_names": [
      "M. Tullius Cicero"
    ],
    "work_urn": "urn:cts:latinLit:phi0474.phi043",
    "titles": [
      "De Republica"
    ],
    "language": "lat",
    "resources": [
      {
        "type": "edition",
        "urn": "urn:cts:latinLit:phi0474.phi043.perse

## 7 - Search for a surface word form <a class="anchor" id="form-search"></a>
##### [Back to ToC](#TOC)

A **surface form** is the spelling that appears in the text. Searching for `μῆνιν` with `kind=form` asks for that written form, not every inflection of the lexical headword.

This request goes directly to Scaife—not CTS and not MCP. Plain Scaife expects a query it understands directly; unlike this project's MCP search tool, this helper does not convert Beta Code to Unicode Greek.

The first broad result may come from an author other than Homer. That is not an error: an unscoped library search covers the indexed collection.

In [9]:
broad_form_search = scaife_search("μῆνιν", kind="form")
print_search_summary("Broad Scaife surface-form search", broad_form_search, limit=5)


Broad Scaife surface-form search
query='μῆνιν', kind='form', total_count=314
[
  {
    "passage_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238",
    "text_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2",
    "author": "Nonnus of Panopolis",
    "work": "Dionysiaca",
    "reference": "Book 45 Line 238",
    "snippet": "μῆνιν ἀλυσκάζοντες ἀθηήτοιο Λυαίου"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0006.tlg004.perseus-grc2:762",
    "text_urn": "urn:cts:greekLit:tlg0006.tlg004.perseus-grc2",
    "author": "Euripides",
    "work": "Heracleidae",
    "reference": "Line 762",
    "snippet": "μῆνιν ἐμᾷ χθονὶ κεύθειν·"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0011.tlg003.perseus-grc2:656",
    "text_urn": "urn:cts:greekLit:tlg0011.tlg003.perseus-grc2",
    "author": "Sophocles",
    "work": "Ajax",
    "reference": "Line 656",
    "snippet": "μῆνιν βαρεῖαν ἐξαλύξωμαι θεᾶς·"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
    "tex

## 8 - Search by lemma <a class="anchor" id="lemma-search"></a>
##### [Back to ToC](#TOC)

A **lemma** is a dictionary headword. A lemma search asks the index for tokens analyzed under that headword, even when the visible text contains an inflected form such as `λόγου`, `λόγῳ`, or `λόγον`.

The code submits the same query, `λόγος`, as both a form and a lemma. Compare the snippets rather than assuming one total must always be larger. Scaife's `total_count` reflects its indexed result model and grouping, so form and lemma totals are useful search metadata but are not a linguistic frequency proof by themselves.

In [10]:
logos_form = scaife_search("λόγος", kind="form")
logos_lemma = scaife_search("λόγος", kind="lemma")

print_search_summary("Surface-form query: λόγος", logos_form, limit=2)
print_search_summary("Lemma query: λόγος", logos_lemma, limit=2)

print("\nCount comparison:")
print(
    json.dumps(
        {
            "form_total_count": logos_form.get("total_count"),
            "lemma_total_count": logos_lemma.get("total_count"),
            "interpretation": "Inspect passages and morphology; do not treat these API totals as directly equivalent token counts.",
        },
        ensure_ascii=False,
        indent=2,
    )
)


Surface-form query: λόγος
query='λόγος', kind='form', total_count=13586
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0031.tlg004.perseus-grc2:1.1",
    "text_urn": "urn:cts:greekLit:tlg0031.tlg004.perseus-grc2",
    "author": "New Testament",
    "work": "Gospel according to John",
    "reference": "Chapter 1 Verse 1",
    "snippet": "ΕΝ ΑΡΧΗ ἦν ὁ λόγος, καὶ ὁ λόγος ἦν πρὸς τὸν θεόν, καὶ θεὸς ἦν ὁ λόγος."
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg1692.tlg005.1st1K-grc1:106",
    "text_urn": "urn:cts:greekLit:tlg1692.tlg005.1st1K-grc1",
    "author": "Speusippus",
    "work": "Fragmenta",
    "reference": "Fragment 106",
    "snippet": "106. Ἀπόδειξις λόγος συλλογιστικὸς ἀληθής· λόγος ἐμφανιστικὸς διὰ προγιγνωσκομένων."
  }
]

Lemma query: λόγος
query='λόγος', kind='lemma', total_count=12052
[
  {
    "passage_urn": "urn:cts:greekLit:tlg2200.tlg00459.opp-grc1:praef",
    "text_urn": "urn:cts:greekLit:tlg2200.tlg00459.opp-grc1",
    "author": "Libanius",
    "work": "Oratio 59",

## 9 - Search for an exact phrase <a class="anchor" id="phrase-search"></a>
##### [Back to ToC](#TOC)

Quotation marks can request an adjacent phrase. The famous opening `"μῆνιν ἄειδε"` is scoped to the *Iliad* work so the result expresses a precise research question: find this phrase in this work.

Scaife also supports operator-like syntax such as exclusion, alternatives, wildcards, and fuzzy suffixes. Those options deserve careful testing and are covered in the advanced search notebook; this introductory example uses only a quoted phrase.

In [11]:
phrase_search = scaife_search(
    '"μῆνιν ἄειδε"',
    kind="form",
    work=ILIAD_WORK,
)
print_search_summary("Quoted phrase within the Iliad", phrase_search, limit=5)


Quoted phrase within the Iliad
query='"μῆνιν ἄειδε"', kind='form', total_count=1
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
    "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 1 Line 1",
    "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"
  }
]


## 10 - Scope a search by author or work <a class="anchor" id="scoped-search"></a>
##### [Back to ToC](#TOC)

The same surface-form query can answer three different questions:

- **unscoped:** where does `μῆνιν` occur anywhere in the indexed library?
- **textgroup-scoped:** where does it occur in texts grouped under Homer?
- **work-scoped:** where does it occur specifically in the *Iliad*?

The `text_group` and `work` values are CTS URNs discovered from the plain CTS inventory, but they are passed as filters to the plain Scaife search API.

In [12]:
homer_form_search = scaife_search(
    "μῆνιν",
    kind="form",
    text_group=HOMER_TEXTGROUP,
)
iliad_form_search = scaife_search(
    "μῆνιν",
    kind="form",
    work=ILIAD_WORK,
)

scope_comparison = {
    "unscoped_library": broad_form_search.get("total_count"),
    "homer_textgroup": homer_form_search.get("total_count"),
    "iliad_work": iliad_form_search.get("total_count"),
}
print(json.dumps(scope_comparison, ensure_ascii=False, indent=2))
print_search_summary("First work-scoped Iliad results", iliad_form_search, limit=5)

{
  "unscoped_library": 314,
  "homer_textgroup": 12,
  "iliad_work": 9
}

First work-scoped Iliad results
query='μῆνιν', kind='form', total_count=9
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
    "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 1 Line 75",
    "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
    "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 5 Line 444",
    "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:16.711",
    "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 16 Line 711",
    "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
  },
  {


## 11 - Demonstrate the CTS and Scaife URN mismatch <a class="anchor" id="urn-mismatch"></a>
##### [Back to ToC](#TOC)

Perseus CTS and Scaife can expose **different edition URNs for the same author and work**. In the current live data, the plain CTS inventory advertises an Iliad edition such as `perseus-grc1`, while the plain Scaife search result uses `perseus-grc2`.

Neither identifier should be silently substituted for the other:

- use the Scaife edition URN with Scaife reader, passage, and highlight routes;
- use an edition discovered from CTS `GetCapabilities` with CTS passage and navigation requests;
- carry the common work URN and citation between the services only after checking them.

The next cell prints the mismatch rather than merely describing it.

In [13]:
first_iliad_hit = iliad_form_search["results"][0]
scaife_passage_urn = first_iliad_hit["passage"]["urn"]
SCAIFE_ILIAD_EDITION = first_iliad_hit["passage"]["text"]["urn"]
scaife_work_urn = SCAIFE_ILIAD_EDITION.rsplit(".", 1)[0]
hit_citation = scaife_passage_urn.rpartition(":")[2]
cts_passage_urn = f"{CTS_ILIAD_EDITION}:{hit_citation}"

urn_comparison = {
    "same_author_and_work": scaife_work_urn == ILIAD_WORK,
    "work_urn": ILIAD_WORK,
    "cts_edition_from_GetCapabilities": CTS_ILIAD_EDITION,
    "scaife_edition_from_search_hit": SCAIFE_ILIAD_EDITION,
    "same_edition_urn": CTS_ILIAD_EDITION == SCAIFE_ILIAD_EDITION,
    "shared_citation": hit_citation,
    "original_scaife_passage_urn": scaife_passage_urn,
    "translated_cts_passage_urn": cts_passage_urn,
    "scaife_snippet": clean_snippet(first_iliad_hit.get("content")),
}

print(json.dumps(urn_comparison, ensure_ascii=False, indent=2))

{
  "same_author_and_work": true,
  "work_urn": "urn:cts:greekLit:tlg0012.tlg001",
  "cts_edition_from_GetCapabilities": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "scaife_edition_from_search_hit": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "same_edition_urn": false,
  "shared_citation": "1.75",
  "original_scaife_passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
  "translated_cts_passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75",
  "scaife_snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
}


## 12 - Navigate around a search hit with plain CTS <a class="anchor" id="navigate-hit"></a>
##### [Back to ToC](#TOC)

A Scaife hit gives us a citation, but CTS navigation must use a CTS-advertised edition. The code therefore:

1. keeps the citation from the Scaife hit;
2. combines it with `CTS_ILIAD_EDITION`, discovered independently from `GetCapabilities`;
3. requests ordered valid references with plain CTS `GetValidReff`;
4. derives the previous and next citations locally;
5. retrieves the corresponding CTS passage and compares it with the Scaife snippet.

This also avoids relying on the live `GetPrevNextUrn` route, which may return malformed HTML.

A successful citation mapping does **not** make the two editions interchangeable. Compare the returned text: editions may differ in punctuation, orthography, markup, or substantive readings. In the current live example, the Scaife and CTS text for `1.75` already shows a small punctuation difference.

In [14]:
valid_references_xml = cts_request("GetValidReff", urn=CTS_ILIAD_EDITION)
valid_references_root = ET.fromstring(valid_references_xml)
valid_urns = [
    element_text(element)
    for element in valid_references_root.iter()
    if local_name(element.tag) == "urn" and element_text(element)
]

if cts_passage_urn not in valid_urns:
    raise RuntimeError(
        "The Scaife citation could not be mapped to the currently advertised CTS edition."
    )

hit_index = valid_urns.index(cts_passage_urn)
previous_urn = valid_urns[hit_index - 1] if hit_index > 0 else None
next_urn = valid_urns[hit_index + 1] if hit_index + 1 < len(valid_urns) else None

cts_passage_xml = cts_request("GetPassage", urn=cts_passage_urn)
cts_passage_root = ET.fromstring(cts_passage_xml)
cts_text_parts = [
    element_text(element)
    for element in cts_passage_root.iter()
    if local_name(element.tag) in {"l", "p", "ab", "seg", "quote", "div"}
    and element_text(element)
]

navigation_result = {
    "previous": previous_urn,
    "current": cts_passage_urn,
    "next": next_urn,
    "scaife_snippet": clean_snippet(first_iliad_hit.get("content")),
    "cts_passage_text": cts_text_parts[0] if cts_text_parts else None,
}

print(json.dumps(navigation_result, ensure_ascii=False, indent=2))

{
  "previous": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.74",
  "current": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75",
  "next": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.76",
  "scaife_snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·",
  "cts_passage_text": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος:"
}


## 13 - Search decision guide and MCP equivalents <a class="anchor" id="decision-guide"></a>
##### [Back to ToC](#TOC)

The following table summarizes both the plain operation demonstrated here and the corresponding project tool. The MCP column is informational only; no MCP tool was called in this notebook.

| Goal | Plain approach demonstrated here | MCP equivalent |
|---|---|---|
| Find partial author names | Download CTS `GetCapabilities`; filter textgroup names | `find_author_names` |
| List an author's works and resources | Filter one CTS textgroup and inspect nested works | `get_author_resources` |
| Find a work by title or URN | Filter CTS work titles and URNs | `get_work_resources` |
| Browse authors and works | Parse and filter the CTS inventory | `list_text_groups` |
| Search an exact written form | Scaife library search with `kind=form` | `search_perseus(search_kind="form")` |
| Search a lexical headword | Scaife library search with `kind=lemma` | `search_perseus(search_kind="lemma")` |
| Search a quoted phrase or operators | Send query syntax directly to Scaife | `search_perseus(preserve_operators=True)` |
| Restrict to an author or work | Pass `text_group` or `work` to Scaife | `search_perseus(author=...)`, `text_group=...`, or `work=...` |
| Search one selected Scaife edition | Use Scaife's reader search route | `search_within_text` |
| Locate matching tokens in a passage | Use Scaife reader search with highlight fields | `get_passage_highlights` |
| Navigate a CTS result | Parse CTS `GetValidReff` and derive neighbors | `get_prev_next_urn` |

The MCP layer adds Unicode/Beta Code normalization, author-name resolution, structured output helpers, caching, and navigation fallbacks. Those conveniences are intentionally absent here so the two upstream services remain visible.

## 14 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Continue with:

- [`01_basic_cts_workflow.ipynb`](01_basic_cts_workflow.ipynb) — review the plain CTS protocol, URN structure, passage retrieval, and reference parsing;
- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) — perform discovery and passage retrieval through an actual FastMCP client;
- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — use the MCP server for Greek search and navigation, including Beta Code normalization;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) — test lemma search, quoted phrases, exclusions, alternatives, wildcards, fuzzy syntax, and author scope;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) — explore server-scoped search, edition-scoped reader search, highlights, caching, and Scaife-native retrieval.

## 15 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook directly uses:

- the [Perseus Digital Library](https://www.perseus.tufts.edu/);
- the plain Perseus [CTS endpoint](https://www.perseus.tufts.edu/hopper/CTS), particularly `GetCapabilities`, `GetValidReff`, and `GetPassage`;
- the plain [Scaife Viewer](https://scaife.perseus.org/) JSON search service;
- the local project overview in the [README](../README.md), used only to relate the demonstrated operations to the MCP tool surface.

Perseus CTS and Scaife are separate live services. Their inventories, edition URNs, result counts, ordering, and response details can change. The notebook therefore discovers resources at runtime and explicitly checks the CTS/Scaife mismatch.

## 16 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

This repository targets **Python 3.11 or newer**. The notebook directly requires:

- `httpx>=0.27.0` for the plain HTTP requests;
- Jupyter/IPython to run the cells.

The `html`, `json`, `re`, `unicodedata`, and `xml.etree.ElementTree` modules are included with Python. No FastMCP dependency is used by the notebook code.

The `%pip install --quiet httpx` cell installs `httpx` into the active kernel. Alternatively, install all repository dependencies with:

```bash
pip install -e .
```

## 17 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.2</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>